# 01 — Dataset Exploration

Inspect YODEP metadata, duration statistics, and speaker balance.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from src.data.yodep_loader import load_yodep_manifest
from src.data.audio_utils import load_audio
from src.utils.logger import setup_logging

setup_logging()

with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

In [ ]:
# Load manifest
raw_dir = Path('../data/yodep/raw')
meta_csv = Path('../data/yodep/metadata.csv')

df = load_yodep_manifest(raw_dir, metadata_csv=meta_csv if meta_csv.exists() else None)
print(f'Total recordings: {len(df)}')
df.head(10)

In [ ]:
# Duration statistics
durations = []
for _, row in df.iterrows():
    try:
        audio, sr = load_audio(Path(row['filepath']), target_sr=16000)
        durations.append(len(audio) / sr)
    except Exception:
        durations.append(None)

df['duration_s'] = durations

print(df.groupby(['language', 'condition'])['duration_s'].describe())

In [ ]:
# Speaker balance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby(['speaker_id', 'language', 'condition']).size().unstack(['language', 'condition']).plot(
    kind='bar', ax=axes[0], title='Recordings per speaker'
)

df.groupby(['language', 'condition'])['duration_s'].mean().unstack('condition').plot(
    kind='bar', ax=axes[1], title='Mean duration by language & condition'
)

plt.tight_layout()
plt.show()